# Benchmark workflow — template
## How to use
1. Edit the **CONFIG** cell (the only cell you need to change).
2. Run all cells top-to-bottom for a fresh benchmark run.
3. Individual sections can be re-run independently if jobs fail.

Diagnostics (section 8) can be run at any time to check how many spectra are
ready per method and which molecules are missing.

In [ ]:
import os, sys, subprocess
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))

from src.workflow.job_submission import (
    submit_slurm_array, submit_qcxms_frag_jobs, run_plotms_for_included,
    submit_crest_jobs, submit_qcxms2_jobs, submit_cfmid_job,
    write_cfmid_idx_smiles, check_crest_status,
)
from src.processing.process_spectra_batch import process_spectra_batch
from src.analysis.run_comparison import run_comparison
from src.analysis.diagnose_spectra import diagnose_spectra

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CONFIG — edit only this cell                                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Short name for this benchmark set — used as the output folder name
DATASET_NAME = "my_dataset"

# Path to your input CSV (must contain a SMILES column)
RAW_CSV = "../data/raw/my_dataset/compounds.csv"

# True  → run TMS derivatisation before simulation (like franklin_tms)
# False → use molecules as-is (like franklin)
DERIVATIZE = False

# ── Methods to run ──────────────────────────────────────────────────────────
RUN_QCXMS  = True
RUN_QCXMS2 = True
RUN_QCXMS2_DFT = False
RUN_NEIMS  = True
RUN_CFMID  = True

# QCxMS variants — choose any subset:
#   "QCxMS_10_ps"       10 ps simulation time
#   "QCxMS_25_ps"       25 ps simulation time
#   "QCxMS_10_ps_iee03" 10 ps with iee=0.3 eV
QCXMS_VARIANTS = ["QCxMS_25_ps"]

# ── Advanced (safe to leave as-is) ──────────────────────────────────────────

# SLURM array specs per QCxMS variant.
# Defaults to "0-{N_MOLS-1}" for any variant not listed here.
GSMD_ARRAY_SPECS = {}  # e.g. {"QCxMS_25_ps": "0-60", "QCxMS_10_ps": "0044,0030"}

# Column name containing SMILES in the input CSV
SMILES_COLUMN = "SMILES"

# Number of molecules (None = auto-detected from processed CSV)
N_MOLS = None

# Molecules where CREST gets stuck — fill in after running the status check
# cell in section 3. Leave empty for the first submission (all are submitted).
CREST_STUCK_FOLDERS = []

# ── Compile-only mode ────────────────────────────────────────────────────────
# True  → skip all job submission; only run spectrum processing, diagnostics,
#         and spectral comparison (sections 7–9).
COMPILE_ONLY = False

# Methods to include in processing, diagnostics, and comparison.
# None → determined by RUN_* flags above.
# Set to a list to analyse a subset without changing RUN_* flags, e.g.:
#   ["QCxMS_25_ps", "QCxMS2", "NEIMS", "CFMID"]
COMPILE_METHODS = None

In [ ]:
# ── Derived paths — no editing needed ──────────────────────────────────────
SRC_ROOT  = os.path.abspath("../src")
DATA_ROOT = os.path.abspath("../data")

PROCESSED_DIR = f"{DATA_ROOT}/processed/{DATASET_NAME}"
SIM_BASE      = f"{DATA_ROOT}/simulation_results/{DATASET_NAME}"

INPUT_CSV      = (f"{PROCESSED_DIR}/{DATASET_NAME}_TMS.csv"
                  if DERIVATIZE else f"{PROCESSED_DIR}/dataset_unique.csv")
SMILES_COL_SIM = "Modified_SMILES" if DERIVATIZE else SMILES_COLUMN

METHODS = (
    (QCXMS_VARIANTS             if RUN_QCXMS      else [])
  + (["QCxMS2"]                 if RUN_QCXMS2     else [])
  + (["QCxMS2_dft"]             if RUN_QCXMS2_DFT else [])
  + (["NEIMS"]                  if RUN_NEIMS       else [])
  + (["CFMID"]                  if RUN_CFMID       else [])
)

# Effective methods for spectrum processing, diagnostics, and comparison
_effective_methods = COMPILE_METHODS if COMPILE_METHODS is not None else METHODS

# Auto-detect molecule count from processed CSV
import pandas as pd
if N_MOLS is None and os.path.exists(INPUT_CSV):
    N_MOLS = len(pd.read_csv(INPUT_CSV))

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SIM_BASE,      exist_ok=True)

print(f"Dataset:      {DATASET_NAME}")
print(f"Derivatize:   {DERIVATIZE}")
print(f"Input CSV:    {INPUT_CSV}  (exists: {os.path.exists(INPUT_CSV)})")
print(f"Sim base:     {SIM_BASE}")
print(f"N_MOLS:       {N_MOLS}")
print(f"Compile-only: {COMPILE_ONLY}")
print(f"Methods:      {_effective_methods}")

## 1. Data preparation

In [ ]:
!python {SRC_ROOT}/processing/remove_duplicate_SMILES_entries.py \
    -i  {RAW_CSV} \
    -o  {PROCESSED_DIR}/dataset_unique.csv \
    --index_map_file {PROCESSED_DIR}/duplicate_mapping.csv \
    --log_file       {PROCESSED_DIR}/dataset_duplicates.csv

In [ ]:
if DERIVATIZE:
    !python {SRC_ROOT}/processing/make_TMS_derivative_251125_v1.py --compare_ref \
        -i {PROCESSED_DIR}/dataset_unique.csv \
        -o {INPUT_CSV}
else:
    print("DERIVATIZE=False — using dataset_unique.csv as INPUT_CSV.")

# Re-detect N_MOLS now that the processed CSV exists
if N_MOLS is None:
    N_MOLS = len(pd.read_csv(INPUT_CSV))
    print(f"N_MOLS = {N_MOLS}")

## 2. QCxMS — directory setup, GS-MD, fragmentation & PlotMS

In [ ]:
if RUN_QCXMS and not COMPILE_ONLY:
    for variant in QCXMS_VARIANTS:
        !python {SRC_ROOT}/processing/make_directories_and_sdfs.py \
            --input_csv   {INPUT_CSV} \
            --output_root {SIM_BASE}/{variant}

In [ ]:
if RUN_QCXMS and not COMPILE_ONLY:
    GSMD_SCRIPTS = {
        "QCxMS_10_ps":       "submit_qcxms_gs_md_10_ps.sh",
        "QCxMS_25_ps":       "submit_qcxms_gs_md_25_ps.sh",
        "QCxMS_10_ps_iee03": "submit_qcxms_gs_md_10_ps_iee03.sh",
    }
    for variant in QCXMS_VARIANTS:
        submit_slurm_array(
            sim_dir     = f"{SIM_BASE}/{variant}",
            script_path = f"{SRC_ROOT}/workflow/{GSMD_SCRIPTS[variant]}",
            array_spec  = GSMD_ARRAY_SPECS.get(variant, f"0-{N_MOLS - 1}"),
        )

In [ ]:
%%capture _frag_out
if RUN_QCXMS and not COMPILE_ONLY:
    FRAG_SCRIPTS = {
        "QCxMS_10_ps":       "submit_qcxms_frag_serial_no_unity.sh",
        "QCxMS_25_ps":       "submit_qcxms_frag_serial_no_unity.sh",
        "QCxMS_10_ps_iee03": "submit_qcxms_frag_serial_no_unity_iee03.sh",
    }
    for variant in QCXMS_VARIANTS:
        submit_qcxms_frag_jobs(
            sim_dir     = f"{SIM_BASE}/{variant}",
            script_path = os.path.abspath(f"{SRC_ROOT}/workflow/{FRAG_SCRIPTS[variant]}"),
            n_mols      = N_MOLS,
        )

In [ ]:
# Check how many trajectories finished (reads stdout of check_qcxms_runs.sh)
if RUN_QCXMS:
    check_script = os.path.abspath(f"{SRC_ROOT}/utils/check_qcxms_runs.sh")
    for variant in QCXMS_VARIANTS:
        print(f"\n=== {variant} ===")
        subprocess.run(["sh", check_script], cwd=f"{SIM_BASE}/{variant}")

In [ ]:
# Run PlotMS for all INCLUDE molecules (requires analysis_decision.txt)
if RUN_QCXMS:
    for variant in QCXMS_VARIANTS:
        print(f"\n--- {variant} ---")
        run_plotms_for_included(wrkdir=f"{SIM_BASE}/{variant}")

## 3. QCxMS2 — CREST conformer search & fragmentation

In [ ]:
if RUN_QCXMS2 and not COMPILE_ONLY:
    !python {SRC_ROOT}/processing/make_directories_and_sdfs.py \
        --input_csv   {INPUT_CSV} \
        --output_root {SIM_BASE}/QCxMS2

In [ ]:
if RUN_QCXMS2 and not COMPILE_ONLY:
    folders = CREST_STUCK_FOLDERS if CREST_STUCK_FOLDERS else [
        f"{i:04d}" for i in range(N_MOLS)
    ]
    submit_crest_jobs(
        sim_dir     = f"{SIM_BASE}/QCxMS2",
        script_path = os.path.abspath(f"{SRC_ROOT}/workflow/submit_batch_crest.sh"),
        folders     = folders,
    )

In [ ]:
# Check CREST status — run after jobs finish.
# Copy any "Not done/stuck" IDs into CREST_STUCK_FOLDERS in CONFIG, then
# re-run the submission cell above.
if RUN_QCXMS2:
    check_crest_status({DATASET_NAME: f"{SIM_BASE}/QCxMS2"})

In [ ]:
if RUN_QCXMS2 and not COMPILE_ONLY:
    submit_qcxms2_jobs(
        base_dir    = f"{SIM_BASE}/QCxMS2",
        bash_script = os.path.abspath(f"{SRC_ROOT}/workflow/submit_qcxms2_job.sh"),
        num_folders = N_MOLS,
    )

## 3b. QCxMS2_dft — wB97X-3c refinement

Uses the same CREST conformer output as QCxMS2, with recommended mixed-level settings:
GFN2-xTB for geometry and IP prescreening; wB97X-3c for barriers and IP refinement.

In [ ]:
if RUN_QCXMS2_DFT and not COMPILE_ONLY:
    from src.workflow.job_submission import submit_qcxms2_jobs
    import shutil
    dft_base = f"{SIM_BASE}/QCxMS2_dft"
    qcxms2_base = f"{SIM_BASE}/QCxMS2"
    # Set up mol dirs and copy crest_best.xyz from QCxMS2
    for mol_id in [f"{i:04d}" for i in range(N_MOLS)]:
        src_xyz = Path(qcxms2_base) / mol_id / "crest_best.xyz"
        dst_dir = Path(dft_base) / mol_id
        if not src_xyz.exists():
            continue
        dst_dir.mkdir(parents=True, exist_ok=True)
        if not (dst_dir / "crest_best.xyz").exists():
            shutil.copy2(src_xyz, dst_dir / "crest_best.xyz")
        for log in ("crest.log", "crest_restart.log"):
            src_log = Path(qcxms2_base) / mol_id / log
            if src_log.exists() and not (dst_dir / log).exists():
                shutil.copy2(src_log, dst_dir / log)
    submit_qcxms2_jobs(
        base_dir       = dft_base,
        bash_script    = os.path.abspath(f"{SRC_ROOT}/workflow/submit_qcxms2_dft_job.sh"),
        method         = "wb97x3c",
        num_folders    = N_MOLS,
    )

## 4. NEIMS

In [ ]:
if RUN_NEIMS and not COMPILE_ONLY:
    !python {SRC_ROOT}/processing/make_directories_and_sdfs.py \
        --input_csv   {INPUT_CSV} \
        --output_root {SIM_BASE}/NEIMS
    submit_slurm_array(
        sim_dir     = f"{SIM_BASE}/NEIMS",
        script_path = f"{SRC_ROOT}/workflow/submit_neims_array.sh",
        array_spec  = f"0-{N_MOLS - 1}",
    )

## 5. CFMID

In [ ]:
if RUN_CFMID and not COMPILE_ONLY:
    !python {SRC_ROOT}/processing/make_directories_and_sdfs.py \
        --input_csv   {INPUT_CSV} \
        --output_root {SIM_BASE}/CFMID
    submit_cfmid_job(
        cfmid_dir   = f"{SIM_BASE}/CFMID",
        script_path = os.path.abspath(f"{SRC_ROOT}/workflow/run_cfmid.sh"),
    )
    write_cfmid_idx_smiles(f"{SIM_BASE}/CFMID")

## 6. Experimental spectra

In [ ]:
if not COMPILE_ONLY:
    subprocess.run([
        "python", f"{SRC_ROOT}/processing/make_inchlkey_sdf_for_nist.py",
        "--input_csv",     INPUT_CSV,
        "--smiles_column", SMILES_COL_SIM,
        "--output_root",   f"{SIM_BASE}/EXP/",
    ])

## 7. Spectrum processing

In [ ]:
process_spectra_batch(SIM_BASE, _effective_methods + ["EXP"])

## 8. Diagnostics — spectra coverage

In [ ]:
diag = diagnose_spectra(SIM_BASE, _effective_methods + ["EXP"], n_mols=N_MOLS)

## 9. Spectral comparison

In [ ]:
run_comparison(SIM_BASE, _effective_methods)

## 9b. Binning sensitivity check

Re-runs the spectral comparison with **10 Da m/z bins** and **basepeak normalisation to 999**
(applied equally to simulated and experimental spectra before scoring).

Results go to `results_bin10bp999/` so they do not overwrite the standard results.
A summary table shows the mean cosine score change vs. 1 Da bins — large positive Δ
for a method suggests that peak position offsets are hurting its score at 1 Da resolution.

In [ ]:
import subprocess, pandas as pd
from pathlib import Path

_BIN_WIDTH = 10
_BASEPEAK  = 999
_suffix    = f"_bin{_BIN_WIDTH}bp{int(_BASEPEAK)}"

# Run binned comparison for all active methods
subprocess.run([
    "python", f"{SRC_ROOT}/analysis/compare_spectra.py",
    "--base_dir",       SIM_BASE,
    "--include_all_peaks",
    "--methods",        *_effective_methods,
    "--bin_width",      str(_BIN_WIDTH),
    "--basepeak",       str(_BASEPEAK),
], check=True)

# Load standard (1 Da) and binned results and compare
def _load_cosines(res_dir, file_suffix=""):
    rows = []
    p = Path(res_dir)
    if not p.exists():
        return pd.DataFrame()
    for mol_dir in sorted(p.iterdir()):
        if mol_dir.name.isdigit():
            csv = mol_dir / f"spectra_all_comparison{file_suffix}.csv"
            if csv.exists():
                df = pd.read_csv(csv)
                df["mol_idx"] = mol_dir.name
                rows.append(df)
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

_std    = _load_cosines(f"{SIM_BASE}/results", "")
_binned = _load_cosines(f"{SIM_BASE}/results{_suffix}", _suffix)

if not _std.empty and not _binned.empty:
    _rows = []
    for _m in _effective_methods:
        _s1  = _std[_std["Method"] == _m]["Cosine"].dropna()
        _s10 = _binned[_binned["Method"] == _m]["Cosine"].dropna()
        _rows.append({
            "Method":          _m,
            "Mean (1 Da)":     _s1.mean()  if len(_s1)  else float("nan"),
            "Mean (10 Da)":    _s10.mean() if len(_s10) else float("nan"),
            "Δ (10 Da−1 Da)":  (_s10.mean() - _s1.mean()) if (len(_s1) and len(_s10)) else float("nan"),
        })
    _summary = pd.DataFrame(_rows).set_index("Method")
    print(_summary.round(1).to_string())
    print(f"\nResults written to {SIM_BASE}/results{_suffix}/")
else:
    print("Could not load results — check that section 9 has been run first.")